In [12]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, SpatialDropout1D, Input,Bidirectional
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import pandas as pd
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [2]:
train_df = pd.read_csv('/content/drive/MyDrive/training_data_binary.csv') #load training data
test_df = pd.read_csv('/content/drive/MyDrive/testing_data_binary.csv') #load testing data

In [4]:
X_train = train_df['prompt']
y_train = train_df['jailbreak']
X_test = test_df['prompt']
y_test = test_df['jailbreak']

In [8]:
max_words = 5000
max_len = 100
tokenizer = Tokenizer(num_words=max_words, lower=True, split=' ')
tokenizer.fit_on_texts(X_train)
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_test_seq = tokenizer.texts_to_sequences(X_test)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)
X_train_seq = tokenizer.texts_to_sequences(X_train)
X_train_pad = pad_sequences(X_train_seq, maxlen=max_len)
X_test_seq = tokenizer.texts_to_sequences(X_test)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len)

In [9]:
model_lstm = Sequential()
model_lstm.add(Embedding(max_words, 100, input_length=max_len))
model_lstm.add(SpatialDropout1D(0.2))
model_lstm.add(Bidirectional(LSTM(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.2)))
model_lstm.add(Bidirectional(LSTM(64, dropout=0.2, recurrent_dropout=0.2)))
model_lstm.add(Dense(128, activation='relu'))
model_lstm.add(Dense(64, activation='relu'))
model_lstm.add(Dense(32, activation='relu'))
model_lstm.add(Dense(1, activation='sigmoid'))

In [11]:
model_lstm.compile(loss='binary_crossentropy', optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), metrics=['accuracy'])
model_lstm.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 100, 100)          500000    
                                                                 
 spatial_dropout1d (Spatial  (None, 100, 100)          0         
 Dropout1D)                                                      
                                                                 
 bidirectional (Bidirection  (None, 100, 256)          234496    
 al)                                                             
                                                                 
 bidirectional_1 (Bidirecti  (None, 128)               164352    
 onal)                                                           
                                                                 
 dense (Dense)               (None, 128)               16512     
                                                        

In [13]:
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
model_checkpoint = ModelCheckpoint('lstm.h5', monitor='val_accuracy', save_best_only=True)

In [14]:
history_lstm = model_lstm.fit(X_train_pad, y_train, epochs=40, batch_size=32, validation_data=(X_test_pad, y_test),callbacks=[early_stopping, model_checkpoint])

Epoch 1/40
171/171 [==============================] - 250s 1s/step - loss: 0.3990 - accuracy: 0.8214 - val_loss: 0.3381 - val_accuracy: 0.8590
Epoch 2/40


/usr/local/lib/python3.10/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


171/171 [==============================] - 226s 1s/step - loss: 0.1659 - accuracy: 0.9425 - val_loss: 0.3920 - val_accuracy: 0.8693
Epoch 3/40
171/171 [==============================] - 219s 1s/step - loss: 0.0775 - accuracy: 0.9756 - val_loss: 0.5710 - val_accuracy: 0.8517
Epoch 4/40
171/171 [==============================] - 219s 1s/step - loss: 0.0490 - accuracy: 0.9831 - val_loss: 0.6542 - val_accuracy: 0.8532
